# SEMTM0051 workshop: convolutional networks and transformers

This workshop goes with the lecture *Beyond all-to-all networks*. All the code is already written, so nothing here is a programming test; what matters is that you run each cell, look carefully at what comes out, and talk through the **Discussion** questions with the people around you, because those questions are where the ideas you will need in the exam are worked out.

There are four parts and a set of practice questions:

1. **Convolution and pooling by hand**: what the operations do, how big the output is, and how many parameters there are.
2. **A small CNN on MNIST**: the network from the lecture, compared with an all-to-all network of the same size, followed by three experiments that show *why* it works better.
3. **From sequences to attention**: recurrent networks and their gradient problem, tokens, and then attention calculated by hand with small matrices.
4. **A tiny transformer**: a complete next-token predictor, small enough to train on a laptop in under a minute.
5. **Practice questions** in the style of the exam.

Everything runs on an ordinary laptop without a GPU. The whole notebook takes three or four minutes to run from top to bottom, most of which is the seven short training runs.

In [ ]:
import math, os, time, urllib.request

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)
print("PyTorch", torch.__version__)

---
# Part 1: convolution and pooling by hand

## 1.1 A convolution is a small detector slid across the image

In an all-to-all layer every unit has its own weight for every pixel, so a unit that has learned to see a vertical bar in the top left of the image knows nothing about vertical bars anywhere else. A convolutional layer instead learns one small set of weights, the **kernel** or **filter**, and applies it at every position. With a $3\times 3$ kernel $w$ and an image $x$, the output at position $(i,j)$ is

$$h_{ij} = \sigma\Big(\beta + \sum_{k=0}^{2}\sum_{l=0}^{2} w_{kl}\, x_{i+k,\,j+l}\Big),$$

that is: lay the kernel on the image with its top-left corner at $(i,j)$, multiply the nine pairs of numbers, add them up, add a bias $\beta$ and apply the nonlinearity $\sigma$. The grid of outputs is called a **feature map**.

You will see the same thing written with the kernel indexed from its centre, as on the exam formula sheet:

$$a_{x,y} = \sum_{\delta_x=-1}^{1}\ \sum_{\delta_y=-1}^{1} h_{x+\delta_x,\ y+\delta_y}\ W_{\delta_x,\delta_y}.$$

This is the same operation; the only difference is whether you label a patch by its corner or by its middle.

Here it is written out with two loops, with no bias and no nonlinearity yet.

In [ ]:
def conv2d(x, w):
    """Slide the k x k kernel w over the image x (stride 1, no padding)."""
    k = w.shape[0]
    n_rows, n_cols = x.shape[0] - k + 1, x.shape[1] - k + 1
    out = np.zeros((n_rows, n_cols))
    for i in range(n_rows):
        for j in range(n_cols):
            patch = x[i:i + k, j:j + k]          # the k x k patch whose corner is at (i, j)
            out[i, j] = np.sum(patch * w)        # multiply element by element and add up
    return out

# a 10 x 10 image containing one vertical bar and one horizontal bar
image = np.zeros((10, 10))
image[1:6, 3] = 1.0
image[8, 4:9] = 1.0

# a kernel that likes a bright vertical line with dark on either side
kernel = np.array([[-1.0, 2.0, -1.0],
                   [-1.0, 2.0, -1.0],
                   [-1.0, 2.0, -1.0]])

feature_map = np.maximum(conv2d(image, kernel), 0)    # ReLU

fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
ax[0].imshow(image, cmap="gray");           ax[0].set_title("image, 10 x 10")
ax[1].imshow(kernel, cmap="bwr", vmin=-2, vmax=2); ax[1].set_title("kernel, 3 x 3")
ax[2].imshow(feature_map, cmap="gray");     ax[2].set_title(f"feature map, {feature_map.shape[0]} x {feature_map.shape[1]}")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.show()
print(feature_map)

> **Discussion 1.** (a) The feature map is $8\times 8$ although the image is $10\times 10$. Why?
> (b) The response is 6 along the middle of the vertical bar and smaller at its two ends. Why?
> (c) The horizontal bar produces almost no response, apart from a weak one at each end. Explain both halves of that sentence.

## 1.2 Move the feature and the response moves with it

Because the same kernel is used everywhere, shifting the input simply shifts the feature map. **Max pooling** then takes the largest value in each non-overlapping $2\times2$ block,

$$p_{ij} = \max_{k,l\in\{0,1\}} h_{2i+k,\ 2j+l},$$

which halves the size of the map and has no weights to learn. The cell below shifts a vertical bar one pixel to the right and compares the feature maps before and after pooling.

In [ ]:
def maxpool2(h):
    """2 x 2 max pooling over non-overlapping blocks."""
    n_rows, n_cols = h.shape[0] // 2, h.shape[1] // 2
    return h[:2 * n_rows, :2 * n_cols].reshape(n_rows, 2, n_cols, 2).max(axis=(1, 3))

bar = np.zeros((10, 10)); bar[1:6, 3] = 1.0
versions = {"original": bar,
            "shifted by 1": np.roll(bar, 1, axis=1),
            "shifted by 2": np.roll(bar, 2, axis=1)}

fig, ax = plt.subplots(3, 3, figsize=(8, 8))
pooled = {}
for col, (name, img) in enumerate(versions.items()):
    h = np.maximum(conv2d(img, kernel), 0)
    pooled[name] = maxpool2(h)
    for row, (arr, label) in enumerate([(img, "image"), (h, "conv + ReLU"), (pooled[name], "after 2 x 2 max pool")]):
        ax[row, col].imshow(arr, cmap="gray", vmin=0, vmax=max(arr.max(), 1))
        ax[row, col].set_title(f"{label}\n{name}" if row == 0 else label, fontsize=9)
        ax[row, col].set_xticks([]); ax[row, col].set_yticks([])
plt.tight_layout(); plt.show()

print("pooled map unchanged by a shift of 1:", np.array_equal(pooled["original"], pooled["shifted by 1"]))
print("pooled map unchanged by a shift of 2:", np.array_equal(pooled["original"], pooled["shifted by 2"]))

> **Discussion 2.** (a) Before pooling, what happens to the feature map when the bar moves one pixel?
> (b) After pooling, a shift of one pixel changed nothing here but a shift of two pixels did. Why? Would a shift of one pixel *always* leave the pooled map unchanged?
> (c) The lecture compared convolution to the *simple cells* and pooling to the *complex cells* found by Hubel and Wiesel. What is the comparison?
> (d) Apart from this tolerance to small shifts, what is the other reason for pooling?

## 1.3 How big is the output?

For an input of width $n$, a kernel of width $k$, padding $p$ (the number of rows of zeros added on each side) and stride $s$ (how far the kernel moves each step), the output width is

$$n_\text{out} = \left\lfloor\frac{n + 2p - k}{s}\right\rfloor + 1 .$$

With stride 1 and no padding this is just $n-k+1$: the map shrinks by $k-1$. Padding a $3\times3$ convolution with $p=1$ keeps the size the same, which is often convenient. A $2\times2$ max pool halves the size. The number of *channels* is separate from all of this: it is simply the number of different filters in the layer.

The cell below follows the two networks from the lecture.

In [ ]:
def conv_out(n, k, stride=1, padding=0):
    return (n + 2 * padding - k) // stride + 1

print("LeNet-5 on a 32 x 32 image")
n = 32;                              print("  input            ", n)
n = conv_out(n, k=5);                print("  5 x 5 conv       ", n)
n = n // 2;                          print("  2 x 2 max pool   ", n)
n = conv_out(n, k=5);                print("  5 x 5 conv       ", n)
n = n // 2;                          print("  2 x 2 max pool   ", n)

print("The small MNIST network on a 28 x 28 image")
n = 28;                              print("  input            ", n)
n = conv_out(n, k=3, padding=1);     print("  3 x 3 conv, p=1  ", n)
n = n // 2;                          print("  2 x 2 max pool   ", n)
n = conv_out(n, k=3, padding=1);     print("  3 x 3 conv, p=1  ", n)
n = n // 2;                          print("  2 x 2 max pool   ", n)

## 1.4 How many parameters?

Each output channel is one detector. It looks at a $k\times k$ patch in *every* input channel, so it has $k\cdot k\cdot C_\text{in}$ weights, plus one bias. With $C_\text{out}$ output channels,

$$\text{parameters} = (k\cdot k\cdot C_\text{in} + 1)\times C_\text{out}.$$

The size of the image does not appear in this formula. An all-to-all layer, `nn.Linear(n_in, n_out)`, has a weight for every pair of units and a bias for every output: $n_\text{in}\, n_\text{out} + n_\text{out}$.

In [ ]:
def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

# first layer of LeNet-5: 3 input channels, 6 output channels, 5 x 5 kernel
conv = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
print("conv layer, by formula :", (5 * 5 * 3 + 1) * 6)
print("conv layer, by PyTorch :", count_parameters(conv))
print("  weight shape", tuple(conv.weight.shape), " bias shape", tuple(conv.bias.shape))

# an all-to-all layer between the same two sets of units
n_in, n_out = 32 * 32 * 3, 28 * 28 * 6
dense = nn.Linear(n_in, n_out)
print(f"all-to-all layer from {n_in} units to {n_out} units: {count_parameters(dense):,}")

> **Discussion 3.** (a) The convolutional layer has 456 parameters and the all-to-all layer between the same units has about 14.5 million. Where exactly does the saving come from? There are two separate things going on.
> (b) If the input image were $320\times320$ instead of $32\times32$, how many parameters would the convolutional layer have?
> (c) Fewer parameters means less memory, but that is not the main benefit. What is?

---
# Part 2: a small CNN on MNIST

## 2.1 The data

MNIST is 60,000 training images and 10,000 test images of handwritten digits, each $28\times28$ pixels in greyscale. The first time you run the next cell it downloads the data, about 10 MB. We load everything into two big tensors, because for a dataset this small that is much faster than reading one image at a time.

In [ ]:
from torchvision import datasets

train_set = datasets.MNIST("data", train=True, download=True)
test_set = datasets.MNIST("data", train=False, download=True)

def prepare(dataset):
    x = dataset.data.float() / 255.0          # pixel values between 0 and 1
    x = (x - 0.1307) / 0.3081                 # subtract the mean, divide by the standard deviation
    return x.unsqueeze(1), dataset.targets    # shape (N, 1, 28, 28): one channel

x_train, y_train = prepare(train_set)
x_test, y_test = prepare(test_set)
print("training images", tuple(x_train.shape), " test images", tuple(x_test.shape))

fig, ax = plt.subplots(1, 10, figsize=(10, 1.4))
for i in range(10):
    ax[i].imshow(x_train[i, 0], cmap="gray"); ax[i].set_title(int(y_train[i])); ax[i].axis("off")
plt.show()

## 2.2 Two networks with about the same number of parameters

The CNN is the one from the lecture: two $3\times3$ convolutions, padded so they keep the size, each followed by a ReLU and a $2\times2$ max pool, then one all-to-all layer to the ten digits. Its rival is an ordinary all-to-all network with one hidden layer of 12 units, chosen so that the two have nearly the same number of parameters.

In [ ]:
def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(16 * 7 * 7, 10),
    )

def make_mlp():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 12), nn.ReLU(),
        nn.Linear(12, 10),
    )

cnn, mlp = make_cnn(), make_mlp()

# follow one image through the CNN, layer by layer
h = x_train[:1]
print(f"{'input':<12}{str(tuple(h.shape)):<20}")
for layer in cnn:
    h = layer(h)
    print(f"{layer.__class__.__name__:<12}{str(tuple(h.shape)):<20}{count_parameters(layer):>6} parameters")

print("\nCNN total       :", count_parameters(cnn))
print("all-to-all total:", count_parameters(mlp))

> **Discussion 4.** (a) Check the three parameter counts for the CNN, 80, 1168 and 7850, using the formulas from Part 1.
> (b) Check the 9550 for the all-to-all network.
> (c) Most of the CNN's parameters are in its final all-to-all layer. Why is that layer so much bigger than the convolutions?
> (d) The network outputs ten numbers, the *logits*, and we never apply a softmax in the model. Where does the softmax happen?

## 2.3 Training

The training loop is the usual one: shuffle the data, go through it in **minibatches** of 64 images, and for each minibatch compute the loss, backpropagate, and let Adam update the weights. Three passes through the data, three **epochs**, is enough.

Notice that the test set is touched only once, at the very end. If we used test accuracy to decide when to stop training, or to choose between designs, the test set would no longer give an honest estimate of how the network does on new data; a separate validation set, split off from the training data, is the thing to use for decisions of that kind.

In [ ]:
def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        return (model(x).argmax(dim=1) == y).float().mean().item()

def train(model, x, y, epochs=3, batch_size=64, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        total = 0.0
        for start in range(0, len(x), batch_size):
            batch = order[start:start + batch_size]
            loss = F.cross_entropy(model(x[batch]), y[batch])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total += loss.item() * len(batch)
        print(f"  epoch {epoch + 1}: mean training loss {total / len(x):.3f}")
    return model

results = {}
for name, model in [("small CNN", cnn), ("all-to-all", mlp)]:
    print(name)
    start = time.time()
    train(model, x_train, y_train)
    results[name] = accuracy(model, x_test, y_test)
    print(f"  test accuracy {100 * results[name]:.1f}%   ({time.time() - start:.0f} s)")

> **Discussion 5.** The two networks have nearly the same number of parameters, saw the same data and were trained in the same way, yet the CNN makes several times fewer mistakes. In terms of what each network has to learn, why?

## 2.4 What did the first layer learn?

The first convolution has eight $3\times3$ filters, which we can simply look at; red is a positive weight and blue a negative one. Below them are the eight feature maps they produce for one test image.

In [ ]:
filters = cnn[0].weight.detach()            # shape (8, 1, 3, 3)
digit = x_test[:1]
with torch.no_grad():
    maps = F.relu(cnn[0](digit))[0]          # shape (8, 28, 28)

fig, ax = plt.subplots(2, 9, figsize=(12, 3))
ax[0, 0].axis("off")
ax[1, 0].imshow(digit[0, 0], cmap="gray"); ax[1, 0].set_title("input", fontsize=9); ax[1, 0].axis("off")
limit = filters.abs().max()
for i in range(8):
    ax[0, i + 1].imshow(filters[i, 0], cmap="bwr", vmin=-limit, vmax=limit)
    ax[0, i + 1].set_title(f"filter {i + 1}", fontsize=9); ax[0, i + 1].axis("off")
    ax[1, i + 1].imshow(maps[i], cmap="gray"); ax[1, i + 1].axis("off")
plt.show()

> **Discussion 6.** (a) Pick two filters and say what each seems to respond to, using its feature map as evidence.
> (b) The lecture described a hierarchy from edges up to whole objects. A unit in the first convolution sees a $3\times3$ patch of the image. How large a patch of the original image can affect one unit in the *second* convolution? Remember that there is a $2\times2$ max pool in between.

## 2.5 Experiment: shift the test images

Neither network has ever seen a digit that is not roughly centred. Here we slide every *test* image a few pixels to the right, without retraining anything. The border of an MNIST image is empty, so nothing is lost by sliding it this far.

In [ ]:
def shift_right(x, pixels):
    return torch.roll(x, shifts=pixels, dims=3)

fig, ax = plt.subplots(1, 4, figsize=(6, 1.8))
for pixels in range(4):
    ax[pixels].imshow(shift_right(x_test[:1], pixels)[0, 0], cmap="gray")
    ax[pixels].set_title(f"shift {pixels}", fontsize=9); ax[pixels].axis("off")
plt.show()

print(f"{'shift':<8}{'small CNN':>12}{'all-to-all':>12}")
for pixels in range(4):
    shifted = shift_right(x_test, pixels)
    print(f"{pixels:<8}{100 * accuracy(cnn, shifted, y_test):>11.1f}%{100 * accuracy(mlp, shifted, y_test):>11.1f}%")

> **Discussion 7.** (a) Why does the all-to-all network fall apart after a shift of only two or three pixels, when to us the digit looks the same?
> (b) Why does the CNN cope much better?
> (c) The CNN is still not perfect: its accuracy also drops by a shift of three. Which part of the CNN is responsible for that?

## 2.6 Experiment: scramble the pixels

Now a stranger experiment. Choose one random rearrangement of the 784 pixel positions and apply the *same* rearrangement to every image, training and test. To us the images become noise, but no information has been lost: every pixel is still there, just in a different, fixed place. We then train fresh copies of both networks on the scrambled images.

In [ ]:
torch.manual_seed(0)
scramble = torch.randperm(28 * 28)

def scrambled(x):
    return x.reshape(-1, 784)[:, scramble].reshape(-1, 1, 28, 28)

fig, ax = plt.subplots(1, 2, figsize=(4, 2))
ax[0].imshow(x_train[0, 0], cmap="gray"); ax[0].set_title("original", fontsize=9); ax[0].axis("off")
ax[1].imshow(scrambled(x_train[:1])[0, 0], cmap="gray"); ax[1].set_title("scrambled", fontsize=9); ax[1].axis("off")
plt.show()

scrambled_results = {}
for name, make in [("small CNN", make_cnn), ("all-to-all", make_mlp)]:
    print(name, "on scrambled pixels")
    torch.manual_seed(0)
    model = train(make(), scrambled(x_train), y_train)
    scrambled_results[name] = accuracy(model, scrambled(x_test), y_test)

print(f"\n{'':<14}{'normal':>10}{'scrambled':>12}")
for name in results:
    print(f"{name:<14}{100 * results[name]:>9.1f}%{100 * scrambled_results[name]:>11.1f}%")

> **Discussion 8.** (a) The all-to-all network does just as well on scrambled images as on normal ones. Why does it not care?
> (b) The CNN loses most of its advantage. Why?
> (c) The lecture said that an all-to-all network "fails to exploit the structure of the data". Use this experiment to say precisely what structure the CNN is exploiting.

## 2.7 Global average pooling

In our CNN, 7,850 of the 9,098 parameters sit in the final all-to-all layer, which reads every position of every feature map separately. Modern CNNs usually do something simpler at the end: **global average pooling** replaces each feature map by a single number, its average over all positions, so that sixteen maps become a vector of sixteen numbers, and only that small vector goes to the classifier.

Here is the same network with the final max pool, the flatten and the big dense layer replaced by a global average pool and a dense layer from 16 numbers to 10 classes.

In [ ]:
def make_gap_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),        # global average pooling: (16, 14, 14) -> (16, 1, 1)
        nn.Flatten(),                   # -> 16 numbers
        nn.Linear(16, 10),
    )

gap_cnn = make_gap_cnn()
print("parameters:", count_parameters(gap_cnn), " (the original CNN has", count_parameters(cnn), ")")
train(gap_cnn, x_train, y_train)

print(f"\n{'shift':<8}{'small CNN':>12}{'CNN with GAP':>14}")
for pixels in range(4):
    shifted = shift_right(x_test, pixels)
    print(f"{pixels:<8}{100 * accuracy(cnn, shifted, y_test):>11.1f}%{100 * accuracy(gap_cnn, shifted, y_test):>13.1f}%")

> **Discussion 9.** (a) What does global average pooling do to a set of feature maps of shape $16\times14\times14$? Check the new parameter count.
> (b) The accuracy of this network hardly changes when the test images are shifted. Why?
> (c) Give two reasons why global average pooling is used before the final classifier of a modern CNN.
> (d) This particular network is much *less* accurate than our original CNN. Why does it work badly here, when it works well at the end of a deep network?

---
# Part 3: from sequences to attention

## 3.1 Recurrent networks and the gradient problem

Text has structure too, but of a different kind: it is a sequence, of any length, in which a word can depend on another word a long way back. The first idea was the recurrent network, which reads one word at a time and carries forward a hidden state $h_t$, using the same weights at every step. To learn that something $k$ steps back matters, the gradient has to travel back through $k$ steps, and it is multiplied by a factor at each one:

$$\frac{\partial h_t}{\partial h_{t-k}} = \prod_{s=t-k+1}^{t} \text{diag}\big(\sigma'_s\big)\, W .$$

The cell below shows what happens to a product of $k$ factors when each is a little smaller, or a little bigger, than one.

In [ ]:
k = np.arange(0, 61)
plt.figure(figsize=(6, 3.2))
plt.semilogy(k, 0.8 ** k, label="each factor 0.8: vanishes")
plt.semilogy(k, 1.0 ** k, "k--", label="each factor 1.0")
plt.semilogy(k, 1.2 ** k, label="each factor 1.2: explodes")
plt.xlabel("number of steps back, k"); plt.ylabel("size of the gradient"); plt.legend(); plt.show()
print(f"after 50 steps:  0.8^50 = {0.8 ** 50:.1e},   1.2^50 = {1.2 ** 50:.1e}")

A vanishing gradient means the network cannot learn long-range dependencies; an exploding gradient means one enormous update can wreck the weights. The usual patch for the second problem is **gradient clipping**: before each update, measure the overall size (the norm) of the gradient, and if it is larger than some threshold $c$, scale the whole gradient down so that its norm is exactly $c$. The direction is kept and only the length changes.

In [ ]:
def clip_by_norm(gradient, c):
    norm = np.linalg.norm(gradient)
    return gradient if norm <= c else gradient * (c / norm)

for g in [np.array([0.3, -0.4]), np.array([300.0, -400.0])]:
    clipped = clip_by_norm(g, c=1.0)
    print(f"gradient {g}  norm {np.linalg.norm(g):7.1f}   ->   clipped {clipped}  norm {np.linalg.norm(clipped):.1f}")

> **Discussion 10.** (a) What is gradient clipping, and why is it used?
> (b) Clipping does nothing for vanishing gradients. What did the LSTM do about those?
> (c) The lecture listed two further problems with recurrent networks that have nothing to do with gradients. What are they?

## 3.2 Tokens and embeddings

A network needs numbers, so text is first cut into pieces called **tokens** and each distinct token is given an integer. The lecture used words as tokens because that is easiest to draw. Real language models use *sub-word* tokens, found automatically from data: common words are a single token, while rarer words are split into several pieces, so that something like "unhappiness" might become "un", "happi", "ness" (the exact split depends on the tokeniser). This keeps the vocabulary to a manageable size, typically tens of thousands of tokens, and means no word is ever out of vocabulary, since in the worst case it can be spelled out from small pieces.

In this workshop we go to the other extreme and use single **characters** as tokens, which gives a vocabulary small enough to train on a laptop. The text is about a megabyte of Shakespeare.

In [ ]:
if not os.path.exists("shakespeare.txt"):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "shakespeare.txt")

text = open("shakespeare.txt").read()
print(f"{len(text):,} characters\n")
print(text[:200])

vocabulary = sorted(set(text))
token_of = {ch: i for i, ch in enumerate(vocabulary)}
char_of = {i: ch for i, ch in enumerate(vocabulary)}
vocab_size = len(vocabulary)

def encode(s): return [token_of[ch] for ch in s]
def decode(tokens): return "".join(char_of[int(t)] for t in tokens)

print("\nvocabulary size:", vocab_size)
print("vocabulary:", repr("".join(vocabulary)))
print("encode('mother') =", encode("mother"))

An **embedding** then turns each token number into a vector. It is nothing more than a table with one row per token, and the rows are learned along with everything else.

In [ ]:
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=4)
tokens = torch.tensor(encode("mother"))
print("tokens:", tokens.tolist())
print("embedded, shape", tuple(embedding(tokens).shape))
print(embedding(tokens).detach().numpy())

> **Discussion 11.** (a) What is a token? Give an example showing that a token need not be a whole word.
> (b) Give one advantage and one disadvantage of using characters as tokens, compared with whole words.
> (c) An embedding layer is equivalent to multiplying a one-hot vector by a matrix. Explain why, and say how many parameters `nn.Embedding(65, 4)` has.

## 3.3 Attention by hand

Attention updates the vector at each position with a weighted average of vectors from the positions up to and including it. Each token $j$, with vector $x_j$, makes three vectors by three learned linear maps:

- a **query** $q_j$: what am I looking for?
- a **key** $k_j$: what do I have to offer?
- a **value** $v_j$: what do I pass on if I am chosen?

Token $i$ scores token $j$ by the dot product $q_i\cdot k_j$ of its query with that token's key, turns its scores into weights with a softmax, and takes the weighted average of the values:

$$a_{ij} = \frac{e^{q_i\cdot k_j}}{\sum_{l\le i} e^{q_i\cdot k_l}},\qquad z_i = \sum_{j\le i} a_{ij}\, v_j .$$

A useful picture is looking something up in a dictionary or a database. You arrive with a query; every entry has a key, which is what your query is compared with, and a value, which is what you get back. The difference is that an ordinary lookup returns the one entry whose key matches exactly, whereas attention is a *soft* lookup: every key matches to some degree, and what comes back is a blend of all the values, weighted by how well each key matched.

Let us do one calculation completely by hand. There are three tokens with two-dimensional vectors, stored as the rows of $X$, and to keep the arithmetic easy the three maps are all the identity, so $q_i=k_i=v_i=x_i$.

In [ ]:
X = np.array([[1.0, 0.0],      # token 1
              [1.0, 1.0],      # token 2
              [0.0, 2.0]])     # token 3

W_Q = W_K = W_V = np.eye(2)
Q, K, V = X @ W_Q, X @ W_K, X @ W_V      # one row per token

S = Q @ K.T                               # S[i, j] = q_i . k_j
print("raw scores S = Q K^T\n", S)

Row $i$ of $S$ holds the scores that token $i$ gives to every token, including the ones that come after it. But the model is being trained to predict the next token at every position, so token $i$ must not be allowed to look at tokens $i+1, i+2,\dots$: those are exactly what it is supposed to be predicting. The **causal mask** removes the entries above the diagonal, by setting those scores to $-\infty$ so that after the softmax their weights are exactly zero.

In [ ]:
allowed = np.tril(np.ones((3, 3), dtype=bool))       # True on and below the diagonal
print("allowed to attend?\n", allowed)

S_masked = np.where(allowed, S, -np.inf)
print("\nmasked scores\n", S_masked)

def softmax_rows(scores):
    e = np.exp(scores - scores.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

A = softmax_rows(S_masked)
print("\nattention weights A (each row adds up to one)\n", A)

Z = A @ V
print("\noutput Z = A V (row i is the new vector for token i)\n", Z)

Check the middle row by hand, since this is precisely the kind of calculation you could be asked to do. Token 2 may attend to tokens 1 and 2, with scores $S_{21}=1$ and $S_{22}=2$:

$$a_{21} = \frac{e^{1}}{e^{1}+e^{2}} = \frac{2.718}{2.718+7.389} = 0.269,\qquad a_{22} = \frac{e^{2}}{e^{1}+e^{2}} = 0.731,$$

$$z_2 = 0.269\,(1,0) + 0.731\,(1,1) = (1.000,\ 0.731).$$

All of this is one line in matrix form, which is how it appears on the formula sheet, with the softmax applied to each row:

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt d}\right)V .$$

The only new thing is the division by $\sqrt d$, where $d$ is the length of the query and key vectors. We ignored it above, as hand calculations usually do; the next cell shows why it is there. It draws a random query and ten random keys of length $d$ and reports the largest of the ten attention weights, averaged over 500 repeats; if the ten tokens shared the attention equally, each weight would be 0.1.

In [ ]:
rng = np.random.default_rng(0)
print(f"{'d':>6}{'largest weight, no scaling':>30}{'largest weight, with 1/sqrt(d)':>34}")
for d in [2, 16, 128, 1024]:
    unscaled, scaled = [], []
    for trial in range(500):                       # average over many random queries
        q = rng.standard_normal(d)                 # one query
        keys = rng.standard_normal((10, d))        # ten keys
        scores = keys @ q
        unscaled.append(softmax_rows(scores[None, :]).max())
        scaled.append(softmax_rows(scores[None, :] / np.sqrt(d)).max())
    print(f"{d:>6}{np.mean(unscaled):>30.3f}{np.mean(scaled):>34.3f}")

> **Discussion 12.** (a) Using the dictionary or database picture, explain the role of the query, the key and the value.
> (b) Which entries of the score matrix does the causal mask remove, and why is that needed when the model is trained to predict the next token?
> (c) Token 1 can only attend to itself. What is its attention weight, and what is its output, whatever the numbers are?
> (d) Calculate the third row of $A$ and of $Z$ by hand and check against the output above.
> (e) From the table: what goes wrong without the $1/\sqrt d$ when $d$ is large, and why would that be bad for training?

## 3.4 Attention has no idea of order

Look again at the formula: the weight $a_{ij}$ depends on the *contents* $q_i$ and $k_j$ and on nothing else. Nowhere does it say how far apart $i$ and $j$ are, or which came first. The cell below feeds the same tokens in a different order.

In [ ]:
def causal_attention(X):
    """Single-head causal self-attention with W_Q = W_K = W_V = I and no scaling."""
    n = len(X)
    S = np.where(np.tril(np.ones((n, n), dtype=bool)), X @ X.T, -np.inf)
    return softmax_rows(S) @ X

X_original = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 2.0], [2.0, 1.0]])    # tokens 1 2 3 4
X_reordered = X_original[[2, 0, 1, 3]]                                     # tokens 3 1 2 4

print("output at the last position, original order :", causal_attention(X_original)[-1])
print("output at the last position, earlier tokens reordered:", causal_attention(X_reordered)[-1])

> **Discussion 13.** (a) The last token receives exactly the same output whichever order the earlier tokens are in. Why?
> (b) Why is this a serious problem for language? Give a pair of sentences that such a model could not tell apart.
> (c) How do transformers provide information about order?

---
# Part 4: a tiny transformer

Now we put the pieces together into a complete model which, at every position, predicts the next character using only that character and the ones before it. It follows the three steps from the lecture: embed, attend, predict.

## 4.1 The training data

The targets are just the text shifted along by one, so any text is training data and every position gives a training example. We keep the last 10% of the text aside as a **validation set**, to check that the model is learning something general about the text and not memorising the training portion.

In [ ]:
data = torch.tensor(encode(text))
split = int(0.9 * len(data))
train_data, validation_data = data[:split], data[split:]

context_length = 64       # how many characters the model sees at once

def get_batch(source, batch_size=32):
    starts = torch.randint(len(source) - context_length - 1, (batch_size,))
    x = torch.stack([source[s:s + context_length] for s in starts])
    y = torch.stack([source[s + 1:s + context_length + 1] for s in starts])
    return x, y

x, y = get_batch(train_data)
print("input shape", tuple(x.shape), " target shape", tuple(y.shape))
print("input :", repr(decode(x[0][:30])))
print("target:", repr(decode(y[0][:30])))

## 4.2 The model

Read this code slowly; the `CausalSelfAttention` class is the hand calculation of section 3.3 with learned maps $W_Q$, $W_K$, $W_V$ in place of the identity, and with an extra leading dimension so that a whole batch of sequences is processed at once.

Two practical ingredients appear that the lecture did not dwell on, and you do not need their details: each block *adds* its result to its input rather than replacing it (a residual connection, the `x = x + ...` lines), and the vectors are rescaled by a `LayerNorm` before each step. Both are there to make deep stacks of blocks train reliably.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.W_Q = nn.Linear(d, d, bias=False)
        self.W_K = nn.Linear(d, d, bias=False)
        self.W_V = nn.Linear(d, d, bias=False)

    def forward(self, x):                                    # x: (batch, T, d)
        T = x.shape[1]
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        S = Q @ K.transpose(1, 2) / math.sqrt(self.d)        # scores, (batch, T, T)
        allowed = torch.tril(torch.ones(T, T, dtype=torch.bool))
        S = S.masked_fill(~allowed, float("-inf"))           # the causal mask
        A = F.softmax(S, dim=-1)                             # each row adds up to one
        self.last_weights = A.detach()                       # kept so we can look at them later
        return A @ V


class Block(nn.Module):
    """Attention moves information between positions; the MLP then works on each position separately."""
    def __init__(self, d):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attention = CausalSelfAttention(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class TinyTransformer(nn.Module):
    def __init__(self, d=64, n_blocks=2, use_positions=True):
        super().__init__()
        self.use_positions = use_positions
        self.token_embedding = nn.Embedding(vocab_size, d)             # step 1: embed
        self.position_embedding = nn.Embedding(context_length, d)
        self.blocks = nn.ModuleList([Block(d) for _ in range(n_blocks)])   # step 2: attend
        self.final_norm = nn.LayerNorm(d)
        self.to_logits = nn.Linear(d, vocab_size)                      # step 3: predict

    def forward(self, tokens):                                         # tokens: (batch, T)
        x = self.token_embedding(tokens)
        if self.use_positions:
            x = x + self.position_embedding(torch.arange(tokens.shape[1]))
        for block in self.blocks:
            x = block(x)
        return self.to_logits(self.final_norm(x))                      # (batch, T, vocab_size)


torch.manual_seed(0)
model = TinyTransformer()
print("parameters:", f"{count_parameters(model):,}")
logits = model(x)
print("input", tuple(x.shape), "-> logits", tuple(logits.shape))

> **Discussion 14.** (a) The output has shape `(32, 64, 65)`. What does each of the three numbers mean?
> (b) Attention is the only place where information moves between positions. What does the MLP in each block do, and in what way is it "like a convolution"?
> (c) Before any training the loss should be close to $\ln 65 \approx 4.17$. Why?

## 4.3 Training

The loop is the same as for the CNN, with three small differences: each minibatch is a random set of chunks of text; we clip the gradient, as in section 3.1; and every 250 steps we also measure the loss on the validation set, which the optimiser never sees.

In [ ]:
def estimate_loss(model, source, n_batches=20):
    model.eval()
    generator_state = torch.get_rng_state()
    torch.manual_seed(123)                      # always the same batches, so the numbers are comparable
    with torch.no_grad():
        losses = []
        for _ in range(n_batches):
            x, y = get_batch(source, batch_size=64)
            losses.append(F.cross_entropy(model(x).view(-1, vocab_size), y.view(-1)).item())
    torch.set_rng_state(generator_state)
    model.train()
    return sum(losses) / len(losses)

def train_transformer(model, steps=2000, lr=3e-3):
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for step in range(steps + 1):
        if step % 250 == 0:
            history.append((step, estimate_loss(model, train_data), estimate_loss(model, validation_data)))
            print(f"  step {step:5d}   training loss {history[-1][1]:.3f}   validation loss {history[-1][2]:.3f}")
        x, y = get_batch(train_data)
        loss = F.cross_entropy(model(x).view(-1, vocab_size), y.view(-1))
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)      # gradient clipping
        optimiser.step()
    return history

start = time.time()
history = train_transformer(model)
print(f"({time.time() - start:.0f} s)")

steps, train_losses, validation_losses = zip(*history)
plt.figure(figsize=(6, 3.2))
plt.plot(steps, train_losses, label="training"); plt.plot(steps, validation_losses, label="validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

> **Discussion 15.** (a) Is this model overfitting? What would you expect to see in the two curves if it were?
> (b) If you wanted to use early stopping, which of the curves would you watch, and why could you not use a test set for that?
> (c) An RNN reading these 64 characters would need 64 steps, one after another. Why can the transformer be trained on all 64 positions at once?

## 4.4 Generating text

To generate, give the model some text, take its predicted probabilities for the next character, **sample** one character from them, add it to the end, and repeat. The model produces one token at a time, and each new token becomes part of the input for the next.

In [ ]:
def generate(model, prompt, n_new=400, seed=1):
    torch.manual_seed(seed)
    model.eval()
    tokens = torch.tensor([encode(prompt)])
    with torch.no_grad():
        for _ in range(n_new):
            logits = model(tokens[:, -context_length:])      # the model sees at most context_length tokens
            probabilities = F.softmax(logits[0, -1], dim=-1) # prediction at the last position only
            next_token = torch.multinomial(probabilities, 1)
            tokens = torch.cat([tokens, next_token[None]], dim=1)
    model.train()
    return decode(tokens[0])

print(generate(model, "ROMEO:\n"))

> **Discussion 16.** This is not Shakespeare, but remember that half a minute ago the model produced uniformly random characters. List some things it has clearly learned about the text. Then think about what it would take to do better.

## 4.5 Looking at the attention weights

Each attention layer saved its most recent weights, so we can see which earlier characters each character attended to. Row $i$ of each picture shows the weights $a_{ij}$ used to update position $i$.

In [ ]:
sample = "First Citizen:\nBefore we proceed"
model.eval()
with torch.no_grad():
    model(torch.tensor([encode(sample)]))
model.train()

labels = [ch if ch not in " \n" else ("_" if ch == " " else "\\n") for ch in sample]
fig, ax = plt.subplots(1, 2, figsize=(13, 6.5))
for b, block in enumerate(model.blocks):
    ax[b].imshow(block.attention.last_weights[0], cmap="viridis", vmin=0, vmax=1)
    ax[b].set_title(f"attention weights, block {b + 1}")
    ax[b].set_xticks(range(len(sample))); ax[b].set_xticklabels(labels, fontsize=8)
    ax[b].set_yticks(range(len(sample))); ax[b].set_yticklabels(labels, fontsize=8)
    ax[b].set_xlabel("attended to (key)"); ax[b].set_ylabel("position being updated (query)")
plt.tight_layout(); plt.show()

> **Discussion 17.** (a) The top-right triangle of each picture is empty. Why?
> (b) The top-left corner is always bright. Why?
> (c) Describe any pattern you can see in how positions choose what to attend to.

## 4.6 Experiment: take away the positions

Section 3.4 argued that attention on its own has no notion of order. Here is the practical consequence: the same model, trained in the same way, but without the positional embedding.

In [ ]:
torch.manual_seed(0)
model_without_positions = TinyTransformer(use_positions=False)
history_without = train_transformer(model_without_positions)

print(f"\nvalidation loss with positions    : {history[-1][2]:.3f}")
print(f"validation loss without positions : {history_without[-1][2]:.3f}\n")
print(generate(model_without_positions, "ROMEO:\n", n_new=300))

> **Discussion 18.** (a) The model without positions is clearly worse. For predicting the next *character*, why is order so important?
> (b) It is nonetheless better than chance, and its output is not a random jumble. What can it still use? Think about the token at the current position, and about what the causal mask lets through.

## 4.7 Why generation can be made fast: the KV cache

When we generated text in section 4.4 we were wasteful: to produce each new character we ran the whole model again on all the characters so far, recomputing everything for the old positions as well as the new one. The next cell checks whether that recomputation ever gives anything new. It runs the model on the first 20 characters of a sequence, then on the first 21, and compares the results at the 20 positions they share.

In [ ]:
model.eval()
tokens = torch.tensor([encode("First Citizen:\nBefore we proceed any further, hear me speak.")])
with torch.no_grad():
    out_20 = model(tokens[:, :20])
    out_21 = model(tokens[:, :21])
model.train()

difference = (out_20 - out_21[:, :20]).abs().max().item()
print("shape with 20 tokens:", tuple(out_20.shape), "  shape with 21 tokens:", tuple(out_21.shape))
print(f"largest difference at the first 20 positions: {difference:.1e}")

Nothing changes: adding a token at the end leaves everything computed at the earlier positions exactly as it was (on some machines you may see a difference of around $10^{-7}$, which is rounding error). So the keys and values of the earlier tokens, in every block, can be computed once and stored; this store is called the **KV cache**. To generate the next token the model then only has to compute the query, key and value of the one new token, compare that query with the stored keys, and average the stored values.

> **Discussion 19.** (a) Why are the vectors at the first 20 positions unaffected by the 21st token? The answer is one of the ingredients of Part 3.
> (b) Why is it the keys and values that are stored, and not the queries?
> (c) Would the cache still be valid if there were no causal mask?

---
# Part 5: practice questions

These are in the style of the exam: short questions that need a few sentences or a short calculation, and one longer calculation. Try them on paper, with a calculator but without the notebook. The formula sheet in the exam gives the softmax and the scaled dot-product attention formula, $\text{Attention}(Q,K,V)=\text{softmax}\big(QK^\top/\sqrt d\big)V$, but not the formulas for output sizes or parameter counts, so make sure you can reconstruct those from understanding what a convolution does.

**P1.** A $28\times28$ image is passed through a convolution with a $3\times3$ kernel, stride 1 and no padding, followed by $2\times2$ max pooling. What are the spatial dimensions after each operation? What would they be if the convolution used padding of 1? And for a $64\times64$ image and a $7\times7$ kernel with no padding?

**P2.** How many trainable parameters, including biases, are there in (a) a convolutional layer with 8 input channels, 32 output channels and a $5\times5$ kernel, and (b) `nn.Linear(in_features=50, out_features=20)`?

**P3.** The last convolutional layer of a network produces 64 feature maps, each $6\times6$, and there are 10 classes. How many parameters does the final classifier have if (a) the maps are flattened and fed to an all-to-all layer, (b) global average pooling is applied first? Say in a sentence what global average pooling does.

**P4.** A convolutional network and an all-to-all network with the same number of parameters are trained on the same images, and the convolutional network does much better. Give two properties of images that the convolutional network exploits.

**P5.** In a recurrent network, why can gradients explode, and what is the usual remedy?

**P6.** A sequence of three tokens has embeddings

$$x_1 = \begin{pmatrix}2\\0\end{pmatrix},\qquad x_2 = \begin{pmatrix}0\\1\end{pmatrix},\qquad x_3 = \begin{pmatrix}1\\1\end{pmatrix}.$$

A single-head causal self-attention layer has $W_Q=W_K=W_V=I$, so $q_i=k_i=v_i=x_i$; ignore the $1/\sqrt d$ scaling.

(a) Write down the $3\times3$ matrix of raw scores $S_{ij}=q_i^\top k_j$.
(b) Which entries does the causal mask remove?
(c) Compute the output of the layer at each of the three positions, to two decimal places.
(d) A fourth token is now appended. Which of your answers to (c) change?

**P7.** Two sentences contain exactly the same words in different orders. A transformer *without* positional embeddings reads each of them and predicts the next word. Explain why the two predictions are the same, and what is added to a real transformer to prevent this.

---
## Summary

- An all-to-all layer has a weight for every pair of units and knows nothing about the structure of its input. A **convolution** uses one small set of weights at every position: $(k\cdot k\cdot C_\text{in}+1)\,C_\text{out}$ parameters, whatever the image size, and an output of width $\lfloor (n+2p-k)/s\rfloor+1$.
- **Max pooling** halves the maps, gives some tolerance to small shifts, and makes later layers see more of the image. **Global average pooling** reduces each map to one number before the classifier.
- A CNN beats an all-to-all network of the same size on images because images are local and translation invariant; scramble the pixels and the advantage disappears.
- Recurrent networks suffer from vanishing and exploding gradients (**gradient clipping** helps with the second), a fixed-size bottleneck, and strictly sequential computation.
- Text is cut into **tokens**, which need not be whole words, and each token is given an **embedding**.
- In **attention** each token's **query** is compared with every allowed token's **key**, a softmax turns the scores into weights, and the output is the weighted sum of the **values**: $\text{softmax}(QK^\top/\sqrt d)\,V$.
- The **causal mask** stops a token from attending to later tokens, which is necessary for next-token prediction, lets all positions be trained at once, and makes the **KV cache** possible.
- Attention has no notion of order by itself, so **positional embeddings** are added to the token embeddings.